In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import os
import shutil
from sklearn.metrics import accuracy_score

%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

2026-04-30 10:28:51.928139: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import scipy.io
import numpy as np

def load_svhn(path):
    data = scipy.io.loadmat(path)
    X = data["X"]
    y = data["y"].reshape(-1)

    X = np.transpose(X, (3,0,1,2))

    y[y == 10] = 0

    return X, y

In [3]:
X_test, y_test   = load_svhn("/home/ncgadmin/DAT255/DAT255-project/SVHN/test_32x32.mat")

X_test  = X_test.astype("float32") / 255.0
y_test  = keras.utils.to_categorical(y_test, 10)

In [4]:
from qkeras.utils import load_qmodel

# This replaces the standard keras load_model
model = load_qmodel('Model_Qkeras_UniformPrecision_pruned.h5')

score = model.evaluate(X_test, y_test)
model.summary()

/home/ncgadmin/miniconda3/envs/hls4ml-tutorial_HGQ+qkeras/lib/python3.10/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer HeNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


814/814 [==============================] - 7s 8ms/step - loss: 0.2107 - accuracy: 0.9405
Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 sequential_1 (Sequential)   (None, 32, 32, 3)         0         
                                                                 
 qconv0 (QConv2D)            (None, 30, 30, 32)        896       
                                                                 
 batch_normalization_3 (Bat  (None, 30, 30, 32)        128       
 chNormalization)                                                
                                                                 
 relu0 (QActivation)         (None, 30, 30, 32)        0         
                                                                 
 pool0 (MaxPooling2D)        (None, 

In [6]:
import keras

inputs = keras.Input(shape=(32, 32, 3), name="hw_input")

x = model.get_layer('qconv0')(inputs)
x = model.get_layer('batch_normalization_3')(x) 
x = model.get_layer('relu0')(x)
x = model.get_layer('pool0')(x)

x = model.get_layer('qconv1')(x)
x = model.get_layer('batch_normalization_4')(x) # Check the exact index in your model.summary()
x = model.get_layer('relu1')(x)
x = model.get_layer('pool1')(x)

x = model.get_layer('qconv2')(x)
x = model.get_layer('batch_normalization_5')(x)
x = model.get_layer('relu2')(x)
x = model.get_layer('pool2')(x)

x = keras.layers.Flatten()(x)

x = model.get_layer('qdense0')(x)
x = model.get_layer('relu3')(x)

#x = model.get_layer('dropout')(x) 

x = model.get_layer('qdense1')(x)
outputs = model.get_layer('softmax')(x)


model_stripped = keras.Model(inputs=inputs, outputs=outputs)

model_stripped.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hw_input (InputLayer)       [(None, 32, 32, 3)]       0         
                                                                 
 qconv0 (QConv2D)            (None, 30, 30, 32)        896       
                                                                 
 batch_normalization_3 (Bat  (None, 30, 30, 32)        128       
 chNormalization)                                                
                                                                 
 relu0 (QActivation)         (None, 30, 30, 32)        0         
                                                                 
 pool0 (MaxPooling2D)        (None, 15, 15, 32)        0         
                                                                 
 qconv1 (QConv2D)            (None, 13, 13, 64)        18496     
                                                             

In [7]:
score = model_stripped.evaluate(X_test, y_test)

RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.

In [8]:
import hls4ml

output_dir ="hls4ml_prj_UniformPrecision"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
 
hls_config = hls4ml.utils.config_from_keras_model(model_stripped, granularity='name')

hls_config['Model']['Strategy'] = 'Resource'
hls_config['Model']['ReuseFactor'] = 16

hls_model = hls4ml.converters.convert_from_keras_model(
    model_stripped,
    backend='Vitis',
    hls_config=hls_config,
    io_type='io_stream',
    #proj_name = proj_name,
    output_dir=output_dir,
    board       = 'kv260',
    part='xck26-sfvc784-2LV-c',
    clock_period='5',
)
hls_model.compile()

/home/ncgadmin/miniconda3/envs/hls4ml-tutorial_HGQ+qkeras/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


In [9]:
for name, layer in hls_model.graph.items():
    print(f"Layer Name: {name}, Type: {layer.class_name}")

Layer Name: hw_input, Type: Input
Layer Name: qconv0, Type: Conv2D
Layer Name: batch_normalization_3, Type: BatchNormalization
Layer Name: relu0, Type: Activation
Layer Name: pool0, Type: Pooling2D
Layer Name: qconv1, Type: Conv2D
Layer Name: batch_normalization_4, Type: BatchNormalization
Layer Name: relu1, Type: Activation
Layer Name: pool1, Type: Pooling2D
Layer Name: qconv2, Type: Conv2D
Layer Name: batch_normalization_5, Type: BatchNormalization
Layer Name: relu2, Type: Activation
Layer Name: pool2, Type: Pooling2D
Layer Name: flatten_1, Type: Reshape
Layer Name: qdense0, Type: Dense
Layer Name: relu3, Type: Activation
Layer Name: qdense1, Type: Dense
Layer Name: softmax, Type: Softmax


In [10]:
hls_model.build(csim=False, synth=True, cosim=False)


****** vitis-run v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Thu Apr 30 10:30:13 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
Sourcing Tcl script '/home/ncgadmin/DAT255/DAT255-project/SVHN/Qkeras/hls4ml_prj_UniformPrecision/build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
Resolution: For help on HLS 200-2182 see docs.amd.com/access/sources/dita/topic?Doc_Version=2025.2%20English&url=ug1448-hls-guidance&resourceid=200-2182.html
INFO: [HLS 200-10] Creating and opening solution '/home/ncgadmin/DAT255/DAT255-project/SVHN/Qkeras/hls4ml_prj_UniformPrecision/myproject_prj'.
INFO: [HLS 200-1510] Running: set_top myproject 
INFO: [HLS 200-1510] Running: add_files firmware/myproject.cpp -cflags -std=c++0x 
INFO: [HLS 200-10] Adding design file 'firmware/myproject.cpp' to the project
INFO: [HLS 

Exception: Build failed for myproject. See logs for details.

In [ ]:
hls4ml.report.read_vivado_report(output_dir)